In [5]:
import pandas as pd
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k

/opt/anaconda3/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [7]:
# Cell 1: imports and corrected helper
import pandas as pd
from pathlib import Path

def build_interactions(behaviors_path: Path) -> pd.DataFrame:
    """
    Parse a MIND behaviors.tsv file into a DataFrame of (user_id, article_id)
    pairs for every click—both from the user’s click history and from
    impressions where label==1.
    The MIND behaviors.tsv format is:
      impression_id \t user_id \t timestamp \t history \t impressions
    """
    records = []
    with behaviors_path.open('r', encoding='utf-8') as f:
        for line in f:
            # split into exactly 5 parts
            parts = line.strip().split('\t')
            if len(parts) != 5:
                # skip malformed lines
                continue

            _, user_id, _, history, impressions = parts

            # 1) clicks in the “history” field
            if history != '-':
                for nid in history.split():
                    records.append((user_id, nid))

            # 2) clicks in the “impressions” field (nid-label)
            for imp in impressions.split():
                nid, label = imp.rsplit('-', 1)
                if label == '1':
                    records.append((user_id, nid))

    return pd.DataFrame(records, columns=['user_id', 'article_id'])


In [9]:
base      = Path('/Users/harshadayiniakula/Desktop/RS')
train_beh = base / 'MINDsmall_train' / 'behaviors.tsv'
dev_beh   = base / 'MINDsmall_dev'   / 'behaviors.tsv'

train_df = build_interactions(train_beh)
dev_df   = build_interactions(dev_beh)

# Sanity checks
print("Train file exists:", train_beh.exists())
print("Dev  file exists:", dev_beh.exists())
print("Train interactions:", train_df.shape)
print("Dev interactions:  ", dev_df.shape)

Train file exists: True
Dev  file exists: True
Train interactions: (5343983, 2)
Dev interactions:   (2473897, 2)


In [11]:
# Cell 3: save to CSV for LightFM ingestion
train_df.to_csv('train_interactions.csv', index=False)
dev_df.to_csv('val_interactions.csv',   index=False)

print(f"Saved {len(train_df)} train interactions → train_interactions.csv")
print(f"Saved {len(dev_df)}   val interactions → val_interactions.csv")


Saved 5343983 train interactions → train_interactions.csv
Saved 2473897   val interactions → val_interactions.csv


In [13]:
val_df_filtered = dev_df.copy()  

# Build the user/item vocab lists
all_users = pd.concat([train_df['user_id'], val_df_filtered['user_id']]).unique().tolist()
all_items = pd.concat([train_df['article_id'], val_df_filtered['article_id']]).unique().tolist()

In [15]:
dataset = Dataset()


In [17]:
news = pd.read_csv(
    'MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=[
        'newid','vertical','subvertical',
        'title','abstract','url',
        'title_entities','abstract_entities'
    ],
    dtype=str
)

In [19]:
verticals = news['vertical'].unique().tolist()


In [21]:
print(verticals)

['lifestyle', 'health', 'news', 'sports', 'weather', 'entertainment', 'autos', 'travel', 'foodanddrink', 'tv', 'finance', 'movies', 'video', 'music', 'kids', 'middleeast', 'northamerica']


In [23]:
dataset.fit(
    users=all_users,
    items=all_items,
    item_features=verticals
)

In [25]:
# Cell 4: Build interaction matrices
train_interactions, _ = dataset.build_interactions(
    train_df[['user_id','article_id']].itertuples(index=False, name=None)
)
val_interactions, _ = dataset.build_interactions(
    val_df_filtered[['user_id','article_id']].itertuples(index=False, name=None)
)


In [26]:
vertical_map = news.set_index('newid')['vertical'].to_dict()

# 2) Create (item_id, [feature_name]) tuples for items in your split
item_feature_tuples = [
    (item, [vertical_map[item]])      # <-- wrap the vertical in a list!
    for item in all_items
    if item in vertical_map
]

# 3) Build the sparse CSR matrix of shape (n_items × 17)
item_features = dataset.build_item_features(item_feature_tuples)

### Model Training

In [45]:
model = LightFM(
    no_components=30,
    loss='warp',        # try 'bpr' or 'logistic' also in HPO
    user_alpha=1e-6,
    item_alpha=1e-6
)

30 latent embedding vectors in the model : this is because we already have 17 explicit features, giving a bit more space will allow it to learn hidden factors or dimensions better.

In [47]:
# Cell 7: Train the model, passing in your new item_features
model.fit(
    train_interactions,
    item_features=item_features,
    epochs=10,
    num_threads=4,
    verbose=True
)

Epoch: 100%|████████████████████████████████████| 10/10 [01:05<00:00,  6.54s/it]


In [48]:
# Cell 8: Evaluate on validation with Precision@10
val_precision = precision_at_k(
    model,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

print(f'Validation Precision@10: {val_precision:.4f}')

Validation Precision@10: 0.0686


In [51]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 1) Precision@10
train_prec = precision_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

In [52]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 2) Recall@10
train_rec = recall_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_rec = recall_at_k(
    model,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 3) AUC Score
train_auc = auc_score(
    model,
    train_interactions,
    item_features=item_features
).mean()

val_auc = auc_score(
    model,
    val_interactions,
    item_features=item_features
).mean()

print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_prec:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


NameError: name 'val_prec' is not defined

In [57]:
print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_precision:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


Train Precision@10: 0.0805    |  Val Precision@10: 0.0686
Train Recall@10:    0.0288    |  Val Recall@10:    0.0269
Train AUC:          0.9773    |  Val AUC:          0.8706


### NO REGULARISATION BASIC MODEL

In [29]:
model1 = LightFM(
    no_components = 30,
    loss = 'warp',
)

In [31]:
model1.fit(
    train_interactions,
    item_features=item_features,
    epochs=20,
    num_threads=4,
    verbose=True
)

Epoch: 100%|████████████████████████████████████| 20/20 [02:01<00:00,  6.10s/it]


In [32]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 1) Precision@10
train_prec = precision_at_k(
    model1,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_prec = precision_at_k(
    model1,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 2) Recall@10
train_rec = recall_at_k(
    model1,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_rec = recall_at_k(
    model1,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 3) AUC Score
train_auc = auc_score(
    model1,
    train_interactions,
    item_features=item_features
).mean()

val_auc = auc_score(
    model1,
    val_interactions,
    item_features=item_features
).mean()

print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_prec:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


Train Precision@10: 0.0843    |  Val Precision@10: 0.0723
Train Recall@10:    0.0303    |  Val Recall@10:    0.0279
Train AUC:          0.9826    |  Val AUC:          0.8663


In [33]:
# Cell: Save your trained LightFM model to disk

import pickle

model_path = "lightfm_model1.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model1, f)

print(f"Saved LightFM model to {model_path}")


Saved LightFM model to lightfm_model1.pkl
